In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

!pip install webdriver-manager


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

# Set up ChromeDriver
service = Service(ChromeDriverManager().install())
options = Options()
options.binary_location = "C:\\Program Files\\Google\\Chrome\\Application\\chrome.exe"

driver = webdriver.Chrome(service=service, options=options)

# Navigate to a website
url = "https://www.example.com"
driver.get(url)

# Example: Find an element on the page
element = driver.find_element(By.NAME, "q")
print(element)

# Close the browser
driver.quit()


In [ ]:
import os
os.getcwd()
#print(os.listdir("C:\Program Files\Google\Chrome\Application"))

In [ ]:
!pip install selenium

In [ ]:

from selenium import webdriver # Navegador
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.chrome.service import Service



In [ ]:
import pandas as pd
import sys
import shutil
import os
import time
import datetime
import pickle


In [ ]:
with open('dizionario_zone_roma.pickle', 'rb') as handle:
    dizionario_zone = pickle.load(handle)

In [ ]:
percorso_download=os.getcwd()
chrome_options = Options()
chrome_options.add_experimental_option("prefs", {
  "download.default_directory": percorso_download,
  "download.prompt_for_download": False,
})

chrome_options.add_argument("--window-size=1920x1080")


In [ ]:
prezzi_zona = pd.read_excel("/content/drive/MyDrive/project-data/x_OLD_Codice/Rapporto Affitti Prezzi.xlsx")

In [ ]:
lista_zone = ["Città Giardino", "Pigneto", "Villa Fiorelli"]

In [ ]:
# Specify the path to the ChromeDriver executable
service = Service("/content/drive/MyDrive/project-data/x_OLD_Codice/chromedriver_win32/chromedriver.exe")

# Create Chrome options
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument('--headless')  # Example option

# Initialize the WebDriver with the service and options
driver = webdriver.Chrome(service=service, options=chrome_options)

In [ ]:
driver.command_executor._commands["send_command"] = ("POST", '/session/$sessionId/chromium/send_command')
params = {'cmd': 'Page.setDownloadBehavior', 'params': {'behavior': 'allow', 'downloadPath': percorso_download}}
command_result = driver.execute("send_command", params)

driver.implicitly_wait(2)
driver.set_script_timeout(900)
driver.set_page_load_timeout(900)

In [ ]:
for nome_zona in lista_zone:

    driver.get("https://www.immobiliare.it/")
    time.sleep(5)

    dict_filtri = {
        "Luogo": "Roma",
        "Zona": nome_zona,

        "Prezzo minimo" : None,
        "Prezzo massimo" : 200000,

        "Superficie mininima" : 40,
        "Superficie massima" : None,

        "Tipo Proprieta": ["Intera proprietà", "A reddito"],
        "Piano": None,
    }

    location = driver.find_element(By.XPATH, "/html/body/section[1]/div/div[2]/nd-search-module/form[1]/div[1]/nd-geography-search/input[1]")
    location.send_keys(dict_filtri["Luogo"])
    time.sleep(10)

    location.send_keys(Keys.ENTER)

    time.sleep(10)

    tutto_il_comune = driver.find_element(By.XPATH, '/html/body/section[1]/div/div[2]/nd-search-module/form[1]/div[1]/nd-geography-search/div[2]/div[2]/ul/li[1]/div/div/label/div[2]')
    tutto_il_comune.click()
    time.sleep(3)

    # SELEZIONO LA ZONA

    i,j = dizionario_zone[dict_filtri["Zona"]]
    count=0

    while count<i:
        html = driver.find_element_by_tag_name('html')
        html.send_keys(Keys.DOWN)
        time.sleep(1)
        count+=1
    time.sleep(3)
    macro_zona_button = driver.find_element(By.XPATH, '/html/body/section[1]/div/div[2]/nd-search-module/form[1]/div[1]/nd-geography-search/div[2]/div[2]/ul/li[{}]/div[1]'.format(i))
    macro_zona = macro_zona_button.text
    arrow = driver.find_element(By.XPATH, '/html/body/section[1]/div/div[2]/nd-search-module/form[1]/div[1]/nd-geography-search/div[2]/div[2]/ul/li[{}]/div[1]/div[2]'.format(i))
    arrow.click()
    sub_obj = driver.find_element_by_xpath('/html/body/section[1]/div/div[2]/nd-search-module/form[1]/div[1]/nd-geography-search/div[2]/div[2]/ul/li[{}]/div[2]/ul/li[{}]/label/div[1]'.format(i,j))
    sub_obj.click()
    time.sleep(1)

    prezzo_vendita_macro_zona = prezzi_zona[prezzi_zona["Zone"]==macro_zona].iloc[:,1].values[0]
    prezzo_affitto_macro_zona = prezzi_zona[prezzi_zona["Zone"]==macro_zona].iloc[:,2].values[0]
    rapporto_affitto_vendita_macro_zona = prezzi_zona[prezzi_zona["Zone"]==macro_zona].iloc[:,3].values[0]

    python_button = driver.find_element(By.XPATH,'/html/body/section[1]/div/div[2]/nd-search-module/form[1]/div[1]/nd-geography-search/div/span/span[1]')
    python_button.click()
    time.sleep(3)

    python_button = driver.find_element(By.XPATH,'/html/body/section[1]/div/div[2]/nd-search-module/form[1]/div[4]/button')
    python_button.click()
    time.sleep(3)

    #CLICCKO SULLA X DEL BANNER (SE LA TROVA)

    time.sleep(10)
    try:
        python_button = driver.find_element(By.XPATH, '/html/body/div[9]/div/div/div/div[1]/button')
        python_button.click()
    except:
        print ("Nessun Banner")

    # APRO I FILTRI DELLA RICERCA

    filtri = driver.find_element(By.XPATH, '/html/body/div[4]/div[1]/div/div/form/fieldset/div/div[2]/div[1]/div[6]/button[1]')
    filtri.click()

    # MODIFICO IL PREZZO

    prezzo = driver.find_element(By.XPATH, '/html/body/div[4]/div[1]/div/div/form/fieldset/div/div[2]/div[1]/div[4]/div/div[1]')
    prezzo.click()

    if dict_filtri["Prezzo minimo"]!= None:
        min_price = driver.find_element(By.XPATH, '/html/body/div[4]/div[1]/div/div/form/fieldset/div/div[2]/div[1]/div[4]/div/div[2]/ul/li[1]/div[1]/input')
        min_price.send_keys(dict_filtri["Prezzo minimo"])

    if dict_filtri["Prezzo massimo"]!= None:
        max_price = driver.find_element(By.XPATH, '/html/body/div[4]/div[1]/div/div/form/fieldset/div/div[2]/div[1]/div[4]/div/div[2]/ul/li[1]/div[2]/input')
        max_price.send_keys(dict_filtri["Prezzo massimo"])


    # MODIFICO I FILTRI DELLA DIMENSIONE

    dim = driver.find_element(By.XPATH, '/html/body/div[4]/div[1]/div/div/form/fieldset/div/div[2]/div[1]/div[5]/div/div[1]')
    dim.click()

    if dict_filtri["Superficie mininima"]!= None:
        min_sup = driver.find_element(By.XPATH, '/html/body/div[4]/div[1]/div/div/form/fieldset/div/div[2]/div[1]/div[5]/div/div[2]/ul/li[1]/div[1]/input')
        min_sup.send_keys(dict_filtri["Superficie mininima"])

    if dict_filtri["Superficie massima"]!= None:
        max_sup = driver.find_element(By.XPATH, '/html/body/div[4]/div[1]/div/div/form/fieldset/div/div[2]/div[1]/div[5]/div/div[2]/ul/li[1]/div[2]/input')
        max_sup.send_keys(dict_filtri["Superficie massima"])


    # FILTRO SU PROPRIETA
    if dict_filtri["Tipo Proprieta"] != None:
        tipo_proprietà = driver.find_element(By.XPATH,'/html/body/div[4]/div[1]/div/div/form/fieldset/div/div[2]/div[2]/div[4]/div/div[1]')
        tipo_proprietà.click()
        numero_opzioni = len(driver.find_elements_by_xpath('/html/body/div[4]/div[1]/div/div/form/fieldset/div/div[2]/div[2]/div[4]/div/div[2]/ul/li'))
        for proprieta in dict_filtri["Tipo Proprieta"]:
            for opzione in range(1, numero_opzioni+1):
                oggetto = driver.find_element_by_xpath('/html/body/div[4]/div[1]/div/div/form/fieldset/div/div[2]/div[2]/div[4]/div/div[2]/ul/li[{}]'.format(opzione))
                if oggetto.text == proprieta:
                    button_proprieta = driver.find_element(By.XPATH, '/html/body/div[4]/div[1]/div/div/form/fieldset/div/div[2]/div[2]/div[4]/div/div[2]/ul/li[{}]/a'.format(opzione))
                    button_proprieta.click()



    # FILTRO SU PIANO
    if dict_filtri["Piano"] != None:
        piano = driver.find_element(By.XPATH,'/html/body/div[4]/div[1]/div/div/form/fieldset/div/div[2]/div[2]/div[5]/div/div')
        piano.click()
        numero_opzioni = len(driver.find_elements_by_xpath('/html/body/div[4]/div[1]/div/div/form/fieldset/div/div[2]/div[2]/div[5]/div/ul/li'))
        for piano in dict_filtri["Piano"]:
            for opzione in range(1, numero_opzioni+1):
                oggetto = driver.find_element_by_xpath('/html/body/div[4]/div[1]/div/div/form/fieldset/div/div[2]/div[2]/div[4]/div/div[2]/ul/li[{}]'.format(opzione))
                if oggetto.text == piano:
                    button_piano = driver.find_element(By.XPATH, '/html/body/div[4]/div[1]/div/div/form/fieldset/div/div[2]/div[2]/div[4]/div/div[2]/ul/li[{}]/a'.format(opzione))
                    button_piano.click()


    cerca_button = driver.find_element(By.XPATH, '/html/body/div[4]/div[1]/div/div/form/fieldset/div/div[2]/div[2]/div[17]/button')
    cerca_button.click()

    time.sleep(10)
    try:
        python_button = driver.find_element(By.XPATH, '/html/body/div[8]/div/div/div/div[1]/button')
        python_button.click()
    except:
        print ("Nessun Banner")

    number_button_pages = len(driver.find_elements(By.XPATH, '/html/body/div[4]/div[2]/div/div[5]/div[4]/ul[2]/li'))

    max_num = 0
    for num in range(1,number_button_pages+1):
        numero_pagina = driver.find_element(By.XPATH, '/html/body/div[4]/div[2]/div/div[5]/div[4]/ul[2]/li[{}]'.format(num))
        if numero_pagina.text != "...":
            max_num = max(int(numero_pagina.text), max_num)

    dict_lista_annunci = {
        "ID": [],
        "Titolo": [],
         "Link": [],
         "Prezzo": [],
         "Locali": [],
         "Dimensione": [],
         "Bagni": [],
         "Piano": [],
         "Descrizione": []}

    for pagina in range(1, max_num+1):
        numero_annunci = len(driver.find_elements_by_xpath('/html/body/div[4]/div[2]/div/div[5]/ul/li'))
        for i in range(1,numero_annunci+1):
            ActionChains(driver).move_to_element(driver.find_element_by_xpath('/html/body/div[4]/div[2]/div/div[5]/ul/li[{}]'.format(i))).perform()
            info_id_annuncio = driver.find_element_by_xpath('/html/body/div[4]/div[2]/div/div[5]/ul/li[{}]'.format(i))
            id_annuncio = info_id_annuncio.get_attribute('data-id')
            if id_annuncio is None:
                continue
            info_titolo_annuncio = driver.find_element_by_xpath('/html/body/div[4]/div[2]/div/div[5]/ul/li[{}]/div[2]/div[1]/p/a'.format(i))
            titolo_annuncio = info_titolo_annuncio.text
            link_annuncio = info_titolo_annuncio.get_attribute('href')
            info_descrizione_annuncio = driver.find_element_by_xpath('/html/body/div[4]/div[2]/div/div[5]/ul/li[{}]/div[2]/div[1]/div'.format(i))
            descrizione_annuncio = info_descrizione_annuncio.text
            descrizione_annuncio = descrizione_annuncio.replace("\n", " ")
            numero_proprieta = len(driver.find_elements_by_xpath('/html/body/div[4]/div[2]/div/div[5]/ul/li[{}]/div[2]/div[1]/ul/li'.format(i)))
            flag_prezzo, flag_locali, flag_dim, flag_bagni, flag_piano = 0,0,0,0,0

            for prop in range(1,numero_proprieta+1):
                info_prop = driver.find_element_by_xpath('/html/body/div[4]/div[2]/div/div[5]/ul/li[{}]/div[2]/div[1]/ul/li[{}]'.format(i, prop))
                text_prop = info_prop.text
                text_prop = text_prop.replace("\n", " ")
                if "€" in text_prop:
                    prezzo_annuncio = text_prop
                    flag_prezzo=1
                elif "locali" in text_prop:
                    locali_annuncio = text_prop
                    flag_locali = 1
                elif "m2" in text_prop:
                    dimensione_annuncio = text_prop
                    flag_dim = 1
                elif "bagni" in text_prop:
                    bagni_annuncio = text_prop
                    flag_bagni = 1
                elif "piano" in text_prop:
                    piano_annuncio = text_prop
                    flag_piano = 1

            if flag_prezzo == 0:
                prezzo_annuncio = "???"
            if flag_locali == 0:
                locali_annuncio = "???"
            if flag_dim == 0:
                dimensione_annuncio = "???"
            if flag_bagni == 0:
                bagni_annuncio = "???"
            if flag_piano == 0:
                piano_annuncio = "???"

            dict_lista_annunci["ID"].append(id_annuncio)
            dict_lista_annunci["Titolo"].append(titolo_annuncio)
            dict_lista_annunci["Link"].append(link_annuncio)
            dict_lista_annunci["Prezzo"].append(prezzo_annuncio)
            dict_lista_annunci["Locali"].append(locali_annuncio)
            dict_lista_annunci["Dimensione"].append(dimensione_annuncio)
            dict_lista_annunci["Bagni"].append(bagni_annuncio)
            dict_lista_annunci["Piano"].append(piano_annuncio)
            dict_lista_annunci["Descrizione"].append(descrizione_annuncio)


        html = driver.find_element_by_tag_name('html')
        html.send_keys(Keys.END)
        time.sleep(1)
        count=0
        while count<3:
            html = driver.find_element_by_tag_name('html')
            html.send_keys(Keys.UP)
            count+=1

        time.sleep(5)

        if pagina==1:
            next_page_button = driver.find_element(By.XPATH, '/html/body/div[4]/div[2]/div/div[5]/div[4]/ul[3]/li[1]/a')
            next_page_button.click()
            time.sleep(10)
        elif pagina < max_num:
            next_page_button = driver.find_element(By.XPATH, '/html/body/div[4]/div[2]/div/div[5]/div[3]/ul[3]/li[1]/a')
            next_page_button.click()
            time.sleep(10)


    DataFrame_Annunci = pd.DataFrame.from_dict(dict_lista_annunci, orient="columns")

    DataFrame_Annunci["Zona"] = dict_filtri["Zona"]

    DataFrame_Annunci["Macro Zona"] = macro_zona

    DataFrame_Annunci["Vendita Euro/m2 Macro Zona"] = prezzo_vendita_macro_zona

    DataFrame_Annunci["Affitto Euro/m2 Macro Zona"] = prezzo_affitto_macro_zona

    DataFrame_Annunci["Rapporto Affitto / Vendita Macro Zona"] = rapporto_affitto_vendita_macro_zona

    DataFrame_Annunci["Prezzo_Pulito"] = DataFrame_Annunci["Prezzo"]\
    .str.replace("da", "")\
    .str.replace("€", "")\
    .str.strip()\
    .str.split(" ")\
    .str[0]\
    .str.replace(".", "")\
    .astype(int)

    DataFrame_Annunci["Superficie_Pulita"] = DataFrame_Annunci["Dimensione"]\
    .str.strip()\
    .str.split(" ")\
    .str[0]\
    .astype(int)

    DataFrame_Annunci["Prezzo Euro/m2"] = DataFrame_Annunci["Prezzo_Pulito"] / DataFrame_Annunci["Superficie_Pulita"]

    DataFrame_Annunci["Differenza prezzo Euro/m2  - Vendita Euro/m2 Macro Zona"] = DataFrame_Annunci["Prezzo Euro/m2"] - DataFrame_Annunci["Vendita Euro/m2 Macro Zona"]

    def make_hyperlink(value):
        return '=HYPERLINK("%s")' % (value)

    DataFrame_Annunci['Link'] = DataFrame_Annunci['Link'].apply(lambda x: make_hyperlink(x))

    cols = list(DataFrame_Annunci.columns.values)
    cols.pop(cols.index('Descrizione'))
    DataFrame_Annunci = DataFrame_Annunci[cols+['Descrizione']]

    nome_file = 'Lista_Annunci_Immobiliare' + "_" + dict_filtri["Zona"] + ".xlsx"

    writer = pd.ExcelWriter(nome_file, engine='xlsxwriter')

    DataFrame_Filtri = pd.DataFrame.from_dict(dict_filtri, orient="index")

    DataFrame_Filtri.to_excel(writer, sheet_name='Filtri', header=False)

    DataFrame_Annunci_Altro = DataFrame_Annunci[~((DataFrame_Annunci["Piano"].str.contains("\?")) | (DataFrame_Annunci["Piano"].str.contains("S")) | (DataFrame_Annunci["Piano"].str.contains("T")) | (DataFrame_Annunci["Piano"].str.contains("1")) | (DataFrame_Annunci["Piano"].str.contains("R")))]

    DataFrame_Annunci_Altro.to_excel(writer, sheet_name='Piani Alti - Annunci', index=False)

    DataFrame_Annunci_Primo = DataFrame_Annunci[(DataFrame_Annunci["Piano"].str.contains("1")) | (DataFrame_Annunci["Piano"].str.contains("R"))]

    DataFrame_Annunci_Primo.to_excel(writer, sheet_name='Primo o Rialzato - Annunci', index=False)

    DataFrame_Annunci_Terra = DataFrame_Annunci[DataFrame_Annunci["Piano"].str.contains("T")]

    DataFrame_Annunci_Terra.to_excel(writer, sheet_name='Terra - Annunci', index=False)

    DataFrame_Annunci_Seminterrato = DataFrame_Annunci[DataFrame_Annunci["Piano"].str.contains("S")]

    DataFrame_Annunci_Seminterrato.to_excel(writer, sheet_name='Seminterrato - Annunci', index=False)

    DataFrame_Annunci_question = DataFrame_Annunci[DataFrame_Annunci["Piano"].str.contains("\?")]

    DataFrame_Annunci_question.to_excel(writer, sheet_name='Piano Sconosciuto - Annunci', index=False)

    DataFrame_Annunci.to_excel(writer, sheet_name='Annunci Completo', index=False)

    writer.save()


In [ ]:
driver = webdriver.Chrome(os.getcwd()+ '.\\chromedriver_win32\\chromedriver.exe', options=chrome_options)
driver.command_executor._commands["send_command"] = ("POST", '/session/$sessionId/chromium/send_command')
params = {'cmd': 'Page.setDownloadBehavior', 'params': {'behavior': 'allow', 'downloadPath': percorso_download}}
command_result = driver.execute("send_command", params)

In [ ]:
driver.implicitly_wait(2)
driver.set_script_timeout(900)
driver.set_page_load_timeout(900)
driver.get("https://www.immobiliare.it/")
time.sleep(5)

In [ ]:
dict_filtri = {
    "Luogo": "Roma",
    "Zona": "Città Giardino",

    "Prezzo minimo" : None,
    "Prezzo massimo" : 200000,

    "Superficie mininima" : 40,
    "Superficie massima" : None,

    "Tipo Proprieta": ["Intera proprietà", "A reddito"],
    "Piano": None,
}

In [ ]:
location = driver.find_element(By.XPATH, "/html/body/section[1]/div/div[2]/nd-search-module/form[1]/div[1]/nd-geography-search/input[1]")
location.send_keys(dict_filtri["Luogo"])
time.sleep(10)

In [ ]:
location.send_keys(Keys.ENTER)

In [ ]:
time.sleep(10)

In [ ]:
tutto_il_comune = driver.find_element(By.XPATH, '/html/body/section[1]/div/div[2]/nd-search-module/form[1]/div[1]/nd-geography-search/div[2]/div[2]/ul/li[1]/div/div/label/div[2]')
tutto_il_comune.click()
time.sleep(3)

In [ ]:
# SELEZIONO LA ZONA

i,j = dizionario_zone[dict_filtri["Zona"]]
count=0

while count<i:
    html = driver.find_element_by_tag_name('html')
    html.send_keys(Keys.DOWN)
    time.sleep(1)
    count+=1
time.sleep(3)
macro_zona_button = driver.find_element(By.XPATH, '/html/body/section[1]/div/div[2]/nd-search-module/form[1]/div[1]/nd-geography-search/div[2]/div[2]/ul/li[{}]/div[1]'.format(i))
macro_zona = macro_zona_button.text
arrow = driver.find_element(By.XPATH, '/html/body/section[1]/div/div[2]/nd-search-module/form[1]/div[1]/nd-geography-search/div[2]/div[2]/ul/li[{}]/div[1]/div[2]'.format(i))
arrow.click()
sub_obj = driver.find_element_by_xpath('/html/body/section[1]/div/div[2]/nd-search-module/form[1]/div[1]/nd-geography-search/div[2]/div[2]/ul/li[{}]/div[2]/ul/li[{}]/label/div[1]'.format(i,j))
sub_obj.click()
time.sleep(1)


In [ ]:
prezzo_vendita_macro_zona = prezzi_zona[prezzi_zona["Zone"]==macro_zona]["Vendita (€/m²)"]
prezzo_affitto_macro_zona = prezzi_zona[prezzi_zona["Zone"]==macro_zona]["Affitto (€/m²)"]
rapporto_affitto_vendita_macro_zona = prezzi_zona[prezzi_zona["Zone"]==macro_zona]["Rapporto Affitto / Vendita"]

'''
for zona in dict_filtri["Zone"]:
    i,j = dizionario_zone[zona]
    count=0

    while count<i:  
        html = driver.find_element_by_tag_name('html')
        html.send_keys(Keys.DOWN)
        time.sleep(1)
        count+=1
    time.sleep(3)
    arrow = driver.find_element(By.XPATH, '/html/body/section[1]/div/div[2]/nd-search-module/form[1]/div[1]/nd-geography-search/div[2]/div[2]/ul/li[{}]/div[1]/div[2]'.format(i))
    arrow.click()
    sub_obj = driver.find_element_by_xpath('/html/body/section[1]/div/div[2]/nd-search-module/form[1]/div[1]/nd-geography-search/div[2]/div[2]/ul/li[{}]/div[2]/ul/li[{}]/label/div[1]'.format(i,j))
    sub_obj.click()
    time.sleep(1)
    #ActionChains(driver).move_to_element(driver.find_element_by_xpath('/html/body/section[1]/div/div[2]/nd-search-module/form[1]/div[1]/nd-geography-search/div[2]/div[2]/ul/li[1]/div/div/label/div[2]'.format(i))).perform()
    html = driver.find_element_by_tag_name('html')
    html.send_keys(Keys.CONTROL + Keys.HOME)
    time.sleep(3)
'''

In [ ]:
python_button = driver.find_element(By.XPATH,'/html/body/section[1]/div/div[2]/nd-search-module/form[1]/div[1]/nd-geography-search/div/span/span[1]')
python_button.click()
time.sleep(3)

In [ ]:
python_button = driver.find_element(By.XPATH,'/html/body/section[1]/div/div[2]/nd-search-module/form[1]/div[4]/button')
python_button.click()
time.sleep(3)

In [ ]:
#CLICCKO SULLA X DEL BANNER (SE LA TROVA)

In [ ]:
time.sleep(10)
try:
    python_button = driver.find_element(By.XPATH, '/html/body/div[9]/div/div/div/div[1]/button')
    python_button.click()
except:
    print ("Nessun Banner")

In [ ]:
# APRO I FILTRI DELLA RICERCA

In [ ]:
filtri = driver.find_element(By.XPATH, '/html/body/div[4]/div[1]/div/div/form/fieldset/div/div[2]/div[1]/div[6]/button[1]')
filtri.click()

In [ ]:
# MODIFICO IL PREZZO

In [ ]:
prezzo = driver.find_element(By.XPATH, '/html/body/div[4]/div[1]/div/div/form/fieldset/div/div[2]/div[1]/div[4]/div/div[1]')
prezzo.click()

In [ ]:
if dict_filtri["Prezzo minimo"]!= None:
    min_price = driver.find_element(By.XPATH, '/html/body/div[4]/div[1]/div/div/form/fieldset/div/div[2]/div[1]/div[4]/div/div[2]/ul/li[1]/div[1]/input')
    min_price.send_keys(dict_filtri["Prezzo minimo"])

In [ ]:
if dict_filtri["Prezzo massimo"]!= None:
    max_price = driver.find_element(By.XPATH, '/html/body/div[4]/div[1]/div/div/form/fieldset/div/div[2]/div[1]/div[4]/div/div[2]/ul/li[1]/div[2]/input')
    max_price.send_keys(dict_filtri["Prezzo massimo"])


In [ ]:
# MODIFICO I FILTRI DELLA DIMENSIONE

In [ ]:
dim = driver.find_element(By.XPATH, '/html/body/div[4]/div[1]/div/div/form/fieldset/div/div[2]/div[1]/div[5]/div/div[1]')
dim.click()

In [ ]:
if dict_filtri["Superficie mininima"]!= None:
    min_sup = driver.find_element(By.XPATH, '/html/body/div[4]/div[1]/div/div/form/fieldset/div/div[2]/div[1]/div[5]/div/div[2]/ul/li[1]/div[1]/input')
    min_sup.send_keys(dict_filtri["Superficie mininima"])

In [ ]:
if dict_filtri["Superficie massima"]!= None:
    max_sup = driver.find_element(By.XPATH, '/html/body/div[4]/div[1]/div/div/form/fieldset/div/div[2]/div[1]/div[5]/div/div[2]/ul/li[1]/div[2]/input')
    max_sup.send_keys(dict_filtri["Superficie massima"])


In [ ]:
# FILTRO SU PROPRIETA
if dict_filtri["Tipo Proprieta"] != None:
    tipo_proprietà = driver.find_element(By.XPATH,'/html/body/div[4]/div[1]/div/div/form/fieldset/div/div[2]/div[2]/div[4]/div/div[1]')
    tipo_proprietà.click()
    numero_opzioni = len(driver.find_elements_by_xpath('/html/body/div[4]/div[1]/div/div/form/fieldset/div/div[2]/div[2]/div[4]/div/div[2]/ul/li'))
    for proprieta in dict_filtri["Tipo Proprieta"]:
        for opzione in range(1, numero_opzioni+1):
            oggetto = driver.find_element_by_xpath('/html/body/div[4]/div[1]/div/div/form/fieldset/div/div[2]/div[2]/div[4]/div/div[2]/ul/li[{}]'.format(opzione))
            if oggetto.text == proprieta:
                button_proprieta = driver.find_element(By.XPATH, '/html/body/div[4]/div[1]/div/div/form/fieldset/div/div[2]/div[2]/div[4]/div/div[2]/ul/li[{}]/a'.format(opzione))
                button_proprieta.click()



In [ ]:
# FILTRO SU PIANO
if dict_filtri["Piano"] != None:
    piano = driver.find_element(By.XPATH,'/html/body/div[4]/div[1]/div/div/form/fieldset/div/div[2]/div[2]/div[5]/div/div')
    piano.click()
    numero_opzioni = len(driver.find_elements_by_xpath('/html/body/div[4]/div[1]/div/div/form/fieldset/div/div[2]/div[2]/div[5]/div/ul/li'))
    for piano in dict_filtri["Piano"]:
        for opzione in range(1, numero_opzioni+1):
            oggetto = driver.find_element_by_xpath('/html/body/div[4]/div[1]/div/div/form/fieldset/div/div[2]/div[2]/div[4]/div/div[2]/ul/li[{}]'.format(opzione))
            if oggetto.text == piano:
                button_piano = driver.find_element(By.XPATH, '/html/body/div[4]/div[1]/div/div/form/fieldset/div/div[2]/div[2]/div[4]/div/div[2]/ul/li[{}]/a'.format(opzione))
                button_piano.click()


In [ ]:
cerca_button = driver.find_element(By.XPATH, '/html/body/div[4]/div[1]/div/div/form/fieldset/div/div[2]/div[2]/div[17]/button')
cerca_button.click()

In [ ]:
time.sleep(10)
try:
    python_button = driver.find_element(By.XPATH, '/html/body/div[8]/div/div/div/div[1]/button')
    python_button.click()
except:
    print ("Nessun Banner")

In [ ]:
number_button_pages = len(driver.find_elements(By.XPATH, '/html/body/div[4]/div[2]/div/div[5]/div[4]/ul[2]/li'))

In [ ]:
max_num = 0
for num in range(1,number_button_pages+1):
    numero_pagina = driver.find_element(By.XPATH, '/html/body/div[4]/div[2]/div/div[5]/div[4]/ul[2]/li[{}]'.format(num))
    if numero_pagina.text != "...":
        max_num = max(int(numero_pagina.text), max_num)

In [ ]:
dict_lista_annunci = {
    "ID": [],
    "Titolo": [],
     "Link": [],
     "Prezzo": [],
     "Locali": [],
     "Dimensione": [],
     "Bagni": [],
     "Piano": [],
     "Descrizione": []}

for pagina in range(1, max_num+1):
    numero_annunci = len(driver.find_elements_by_xpath('/html/body/div[4]/div[2]/div/div[5]/ul/li'))
    for i in range(1,numero_annunci+1):
        ActionChains(driver).move_to_element(driver.find_element_by_xpath('/html/body/div[4]/div[2]/div/div[5]/ul/li[{}]'.format(i))).perform()
        info_id_annuncio = driver.find_element_by_xpath('/html/body/div[4]/div[2]/div/div[5]/ul/li[{}]'.format(i))
        id_annuncio = info_id_annuncio.get_attribute('data-id')
        if id_annuncio is None:
            continue
        info_titolo_annuncio = driver.find_element_by_xpath('/html/body/div[4]/div[2]/div/div[5]/ul/li[{}]/div[2]/div[1]/p/a'.format(i))
        titolo_annuncio = info_titolo_annuncio.text
        link_annuncio = info_titolo_annuncio.get_attribute('href')
        info_descrizione_annuncio = driver.find_element_by_xpath('/html/body/div[4]/div[2]/div/div[5]/ul/li[{}]/div[2]/div[1]/div'.format(i))
        descrizione_annuncio = info_descrizione_annuncio.text
        descrizione_annuncio = descrizione_annuncio.replace("\n", " ")
        numero_proprieta = len(driver.find_elements_by_xpath('/html/body/div[4]/div[2]/div/div[5]/ul/li[{}]/div[2]/div[1]/ul/li'.format(i)))
        flag_prezzo, flag_locali, flag_dim, flag_bagni, flag_piano = 0,0,0,0,0

        for prop in range(1,numero_proprieta+1):
            info_prop = driver.find_element_by_xpath('/html/body/div[4]/div[2]/div/div[5]/ul/li[{}]/div[2]/div[1]/ul/li[{}]'.format(i, prop))
            text_prop = info_prop.text
            text_prop = text_prop.replace("\n", " ")
            if "€" in text_prop:
                prezzo_annuncio = text_prop
                flag_prezzo=1
            elif "locali" in text_prop:
                locali_annuncio = text_prop
                flag_locali = 1
            elif "m2" in text_prop:
                dimensione_annuncio = text_prop
                flag_dim = 1
            elif "bagni" in text_prop:
                bagni_annuncio = text_prop
                flag_bagni = 1
            elif "piano" in text_prop:
                piano_annuncio = text_prop
                flag_piano = 1

        if flag_prezzo == 0:
            prezzo_annuncio = "???"
        if flag_locali == 0:
            locali_annuncio = "???"
        if flag_dim == 0:
            dimensione_annuncio = "???"
        if flag_bagni == 0:
            bagni_annuncio = "???"
        if flag_piano == 0:
            piano_annuncio = "???"

        dict_lista_annunci["ID"].append(id_annuncio)
        dict_lista_annunci["Titolo"].append(titolo_annuncio)
        dict_lista_annunci["Link"].append(link_annuncio)
        dict_lista_annunci["Prezzo"].append(prezzo_annuncio)
        dict_lista_annunci["Locali"].append(locali_annuncio)
        dict_lista_annunci["Dimensione"].append(dimensione_annuncio)
        dict_lista_annunci["Bagni"].append(bagni_annuncio)
        dict_lista_annunci["Piano"].append(piano_annuncio)
        dict_lista_annunci["Descrizione"].append(descrizione_annuncio)
        #lista_annunci = driver.find_element(By.XPATH, '/html/body/div[4]/div[2]/div/div[5]/ul/li[{}]'.format(i))
        #dict_lista_annunci["Pagina: {} - Annuncio: {}".format(pagina, i)]=lista_annunci
        #print(lista_annunci.text)
        #print ("\n\n")


    html = driver.find_element_by_tag_name('html')
    html.send_keys(Keys.END)
    time.sleep(1)
    count=0
    while count<3:
        html = driver.find_element_by_tag_name('html')
        html.send_keys(Keys.UP)
        count+=1

    time.sleep(5)

    if pagina==1:
        next_page_button = driver.find_element(By.XPATH, '/html/body/div[4]/div[2]/div/div[5]/div[4]/ul[3]/li[1]/a')
        next_page_button.click()
        time.sleep(10)
    elif pagina < max_num:
        next_page_button = driver.find_element(By.XPATH, '/html/body/div[4]/div[2]/div/div[5]/div[3]/ul[3]/li[1]/a')
        next_page_button.click()
        time.sleep(10)


In [ ]:
DataFrame_Annunci = pd.DataFrame.from_dict(dict_lista_annunci, orient="columns")

In [ ]:
DataFrame_Annunci["Zona"] = dict_filtri["Zona"]

In [ ]:
DataFrame_Annunci["Macro Zona"] = macro_zona

In [ ]:
DataFrame_Annunci["Vendita (€/m²) Macro Zona"] = prezzo_vendita_macro_zona

In [ ]:
DataFrame_Annunci["Affitto (€/m²) Macro Zona"] = prezzo_affitto_macro_zona

In [ ]:
DataFrame_Annunci["Rapporto Affitto / Vendita Macro Zona"] = rapporto_affitto_vendita_macro_zona

In [ ]:
DataFrame_Annunci["Prezzo_Pulito"] = DataFrame_Annunci["Prezzo"]\
.str.replace("da", "")\
.str.replace("€", "")\
.str.strip()\
.str.split(" ")\
.str[0]\
.str.replace(".", "")\
.astype(int)

In [ ]:
DataFrame_Annunci["Superficie_Pulita"] = DataFrame_Annunci["Dimensione"]\
.str.strip()\
.str.split(" ")\
.str[0]\
.astype(int)

In [ ]:
DataFrame_Annunci["Prezzo €/m²"] = DataFrame_Annunci["Prezzo_Pulito"] / DataFrame_Annunci["Superficie_Pulita"]

In [ ]:
DataFrame_Annunci["Prezzo €/m²"]

In [ ]:
def make_hyperlink(value):
    return '=HYPERLINK("%s")' % (value)

DataFrame_Annunci['Link'] = DataFrame_Annunci['Link'].apply(lambda x: make_hyperlink(x))


'''
nome_file = 'Lista_Annunci_Immobiliare'
for zona in dict_filtri["Zone"]:
    nome_file = nome_file + "_" + zona
    
nome_file = nome_file + ".xlsx"
'''

In [ ]:
nome_file = 'Lista_Annunci_Immobiliare' + "_" + dict_filtri["Zona"] + ".xlsx"

In [ ]:
writer = pd.ExcelWriter(nome_file, engine='xlsxwriter')

In [ ]:
DataFrame_Filtri = pd.DataFrame.from_dict(dict_filtri, orient="index")

In [ ]:
DataFrame_Filtri.to_excel(writer, sheet_name='Filtri', header=False)

In [ ]:
DataFrame_Annunci_Altro = DataFrame_Annunci[~((DataFrame_Annunci["Piano"].str.contains("\?")) | (DataFrame_Annunci["Piano"].str.contains("S")) | (DataFrame_Annunci["Piano"].str.contains("T")) | (DataFrame_Annunci["Piano"].str.contains("1")) | (DataFrame_Annunci["Piano"].str.contains("R")))]

In [ ]:
DataFrame_Annunci_Altro.to_excel(writer, sheet_name='Piani Alti - Annunci', index=False)

In [ ]:
DataFrame_Annunci_Primo = DataFrame_Annunci[(DataFrame_Annunci["Piano"].str.contains("1")) | (DataFrame_Annunci["Piano"].str.contains("R"))]

In [ ]:
DataFrame_Annunci_Primo.to_excel(writer, sheet_name='Primo o Rialzato - Annunci', index=False)

In [ ]:
DataFrame_Annunci_Terra = DataFrame_Annunci[DataFrame_Annunci["Piano"].str.contains("T")]

In [ ]:
DataFrame_Annunci_Terra.to_excel(writer, sheet_name='Terra - Annunci', index=False)

In [ ]:
DataFrame_Annunci_Seminterrato = DataFrame_Annunci[DataFrame_Annunci["Piano"].str.contains("S")]

In [ ]:
DataFrame_Annunci_Seminterrato.to_excel(writer, sheet_name='Seminterrato - Annunci', index=False)

In [ ]:
DataFrame_Annunci_question = DataFrame_Annunci[DataFrame_Annunci["Piano"].str.contains("\?")]

In [ ]:
DataFrame_Annunci_question.to_excel(writer, sheet_name='Piano Sconosciuto - Annunci', index=False)

In [ ]:
DataFrame_Annunci.to_excel(writer, sheet_name='Annunci Completo', index=False)

In [ ]:
writer.save()

In [ ]:
#STOP HERE!!!